In [ ]:
from google.colab import files
uploaded = files.upload()

Saving Amazon_Reviews.csv to Amazon_Reviews (1).csv


In [ ]:
import pandas as pd

reviews_df = pd.read_csv(
    "Amazon_Reviews (1).csv",
    engine="python",
    on_bad_lines="skip"
)

reviews_df.columns


Index(['Reviewer Name', 'Profile Link', 'Country', 'Review Count',
       'Review Date', 'Rating', 'Review Title', 'Review Text',
       'Date of Experience'],
      dtype='object')

In [ ]:
# keep only needed columns
df = reviews_df[[
    "Review Text",
    "Rating",
    "Review Date",
    "Country"
]]

# rename to clean names
df = df.rename(columns={
    "Review Text": "review_text",
    "Rating": "rating",
    "Review Date": "review_date",
    "Country": "country"
})

df.head()


,review_text,rating,review_date,country
0,"I registered on the website, tried to order a ...",Rated 1 out of 5 stars,2024-09-16T13:44:26.000Z,US
1,Had multiple orders one turned up and driver h...,Rated 1 out of 5 stars,2024-09-16T18:26:46.000Z,GB
2,I informed these reprobates that I WOULD NOT B...,Rated 1 out of 5 stars,2024-09-16T21:47:39.000Z,GB
3,I have bought from Amazon before and no proble...,Rated 1 out of 5 stars,2024-09-17T07:15:49.000Z,AU
4,If I could give a lower rate I would! I cancel...,Rated 1 out of 5 stars,2024-09-16T18:37:17.000Z,GB


In [ ]:
# drop empty reviews
df = df[df["review_text"].notna()]

# convert rating to numeric
df["rating"] = pd.to_numeric(df["rating"], errors="coerce")

# convert date
df["review_date"] = pd.to_datetime(df["review_date"], errors="coerce")

df.shape


(21055, 4)

In [ ]:
# basic text cleanup
df["review_text"] = (
    df["review_text"]
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

df["review_text"].head()


,review_text
0,"i registered on the website, tried to order a ..."
1,had multiple orders one turned up and driver h...
2,i informed these reprobates that i would not b...
3,i have bought from amazon before and no proble...
4,if i could give a lower rate i would! i cancel...


In [ ]:
!pip install bertopic sentence-transformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 3.0 MB/s eta 0:00:00


In [ ]:
from bertopic import BERTopic

# Initialize the BERTopic model
# This model will be used to discover common themes (topics) in review text
# language="english" tells the model the text is in English
# calculate_probabilities=True allows us to see how strongly a review belongs to a topic
# verbose=True prints progress logs while the model runs

topic_model = BERTopic(
    language="english",
    calculate_probabilities=True,
    verbose=True
)

# Fit the topic model on the review text
# This step:
# 1. Converts each review into numerical embeddings
# 2. Groups similar reviews together
# 3. Assigns a topic ID to each review
# 4. Extracts keywords that describe each topic

topics, probs = topic_model.fit_transform(df["review_text"])


/usr/local/lib/python3.12/dist-packages/hdbscan/robust_single_linkage_.py:175: SyntaxWarning: invalid escape sequence '\{'
  $max \{ core_k(a), core_k(b), 1/\alpha d(a,b) \}$.
2026-01-16 01:54:35,439 - BERTopic - Embedding - Transforming documents to embeddings.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/658 [00:00<?, ?it/s]

2026-01-16 02:11:12,754 - BERTopic - Embedding - Completed ✓
2026-01-16 02:11:12,758 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-01-16 02:12:00,923 - BERTopic - Dimensionality - Completed ✓
2026-01-16 02:12:00,925 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-01-16 02:12:34,840 - BERTopic - Cluster - Completed ✓
2026-01-16 02:12:34,858 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-01-16 02:12:37,412 - BERTopic - Representation - Completed ✓


In [ ]:
# attach topic IDs to the dataframe
df["topic"] = topics

# how many topics were discovered
df["topic"].nunique()


140

In [ ]:
# view topic summary with keywords
topic_info = topic_model.get_topic_info()
topic_info.head(10)


,Topic,Count,Name,Representation,Representative_Docs
0,-1,10939,-1_and_to_the_amazon,"[and, to, the, amazon, they, for, it, is, have...","[i'm writing what i have tried to write twice,..."
1,0,1360,0_driver_door_drivers_parcel,"[driver, door, drivers, parcel, delivery, deli...",[why do the amazon drivers lie ? we had a parc...
2,1,896,1_account_my_card_email,"[account, my, card, email, bank, me, to, phone...",[policies are built for entrapment. after frau...
3,2,630,2_text_found_review_not,"[text, found, review, not, , , , , , ]","[review text not found, review text not found,..."
4,3,581,3_reviews_review_product_negative,"[reviews, review, product, negative, amazon, o...",[“we apologize but amazon has noticed some unu...
5,4,377,4_prime_day_shipping_delivery,"[prime, day, shipping, delivery, days, next, p...",[i have been a prime member for many years. th...
6,5,331,5_books_book_kindle_author,"[books, book, kindle, author, of, from, the, i...","[amazon is my go-to spot for all of my books, ..."
7,6,327,6_delivery_date_day_order,"[delivery, date, day, order, on, it, be, deliv...",[i am waiting for an order that was supposed t...
8,7,287,7_english_customer_speak_service,"[english, customer, speak, service, understand...",[i just spent an hour with amazon customer ser...
9,8,228,8_star_stars_give_zero,"[star, stars, give, zero, could, would, if, on...",[i have to give one star because there is no o...


We summarized the topics discovered by the BERTopic model. Each topic represents a recurring theme or issue found in the customer reviews. The table shows the topic ID assigned by the model, the number of reviews in each topic, an automatically generated topic name based on important keywords, example keywords that describe the topic, and sample reviews that best represent that topic. Topic -1 represents generic or outlier reviews that do not strongly belong to any specific theme. This step converts unstructured review text into structured issue categories that can be used for analysis, reporting, and business insights.

In [ ]:
!pip install transformers


In [ ]:
from transformers import pipeline

# load a pretrained sentiment analysis model
sentiment_model = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

# run sentiment on reviews
df["sentiment"] = df["review_text"].apply(
    lambda x: sentiment_model(x[:512])[0]["label"]
)

df[["review_text", "sentiment"]].head()


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Device set to use cpu


,review_text,sentiment
0,"i registered on the website, tried to order a ...",NEGATIVE
1,had multiple orders one turned up and driver h...,NEGATIVE
2,i informed these reprobates that i would not b...,NEGATIVE
3,i have bought from amazon before and no proble...,NEGATIVE
4,if i could give a lower rate i would! i cancel...,NEGATIVE


In [16]:
# topic vs sentiment summary
topic_sentiment = (
    df.groupby(["topic", "sentiment"])
      .size()
      .reset_index(name="count")
      .sort_values("count", ascending=False)
)

topic_sentiment.head(10)


,topic,sentiment,count
0,-1,NEGATIVE,8190
1,-1,POSITIVE,2749
2,0,NEGATIVE,1298
4,1,NEGATIVE,875
6,2,NEGATIVE,630
7,3,NEGATIVE,514
9,4,NEGATIVE,360
13,6,NEGATIVE,317
15,7,NEGATIVE,271
17,8,NEGATIVE,202


In [17]:
df.to_csv("reviews_with_topics_and_sentiment.csv", index=False)
topic_info.to_csv("topic_summary.csv", index=False)
topic_sentiment.to_csv("topic_sentiment_summary.csv", index=False)
